# Project Buren — Colab GRPO Training

**Before you start:** **Runtime → Change runtime type → GPU (T4+).** Set **`HF_TOKEN`** in Colab secrets (or `os.environ`) if you use gated Hugging Face models.

Run cells **1 → 2 → 3 → 4 → …** in order. **Do not** wipe `/content/buren-env` every session unless you want a fresh clone (cell 3 no longer deletes the folder by default).

**Optional:** run the smoke-test cell after the server cell to verify HTTP + tokenizer + TRL imports before full training.

In [ ]:
# 1) Install dependencies
# - protobuf: Colab often has 3.20.x; wandb/grpc/google need 5.x (<6 for grpcio-status).
# - peft / accelerate / bitsandbytes: required for train.py HF fallback and useful for Unsloth stacks.
# - sentencepiece: many chat models (e.g. Qwen) need it for tokenizers.
# - Omit pip "torch": Colab already ships CUDA torch; reinstalling via pip can break GPU.
# - Unsloth: installed with --no-deps from git HEAD so it doesn't overwrite Colab's transformers.
#   If Unsloth fails to load (AcceleratorError on T4), train.py auto-falls back to HF+PEFT.
import os
try:
    from google.colab import userdata

    _tok = userdata.get("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok
except Exception:
    pass

!pip install -q "protobuf>=5.29.1,<6" \
  "openenv-core>=0.2.3" "fastapi>=0.115" "uvicorn[standard]>=0.30" "pydantic>=2.7" \
  "trl>=0.25" "peft>=0.11" "accelerate>=1.0" "bitsandbytes>=0.45" \
  "sentencepiece" "huggingface_hub>=0.27" "matplotlib>=3.9" \
  "numpy>=2.0" "requests>=2.32" "datasets"

# Install Unsloth from git HEAD with --no-deps to avoid version conflicts with Colab's transformers.
# If the pypi version crashes on your GPU, this git version is more up-to-date.
!pip install -q --no-deps \
  "unsloth @ git+https://github.com/unslothai/unsloth.git" \
  "unsloth-zoo @ git+https://github.com/unslothai/unsloth-zoo.git"

In [ ]:
# 2) Mount Google Drive (optional — for saving checkpoints)
import os

try:
    from google.colab import drive

    drive.mount("/content/drive")
    SAVE_ROOT = "/content/drive/MyDrive/buren-checkpoints"
except ImportError:
    SAVE_ROOT = "/content/buren-checkpoints"
os.makedirs(SAVE_ROOT, exist_ok=True)
print("SAVE_ROOT =", SAVE_ROOT)

In [ ]:
# 3) Get buren-env into /content and on sys.path
import os, sys

os.chdir("/content")
BUREN_ROOT = "/content/buren-env"  # change if you uploaded elsewhere

# First-time only: clone (uncomment and set your repo URL). Do NOT rm -rf every run.
# !git clone https://github.com/<your-org>/buren-env.git

if not os.path.isfile(os.path.join(BUREN_ROOT, "server", "app.py")):
    raise FileNotFoundError(
        f"Project not found at {BUREN_ROOT}. Upload the folder or git clone, then re-run this cell."
    )

os.chdir(BUREN_ROOT)
sys.path.insert(0, BUREN_ROOT)
print("OK:", BUREN_ROOT)

In [ ]:
# 4) Start Buren server in background
import subprocess, sys, time
from client.client import BurenClient

subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server.app:app", "--host", "0.0.0.0", "--port", "7860"],
    cwd=BUREN_ROOT,
    env={**os.environ, "PYTHONPATH": BUREN_ROOT},
)
time.sleep(5)
c = BurenClient("http://127.0.0.1:7860")
print(c.health())

In [ ]:
# 4b) Smoke test — server, client, env API, tokenizer, TRL import (~1–3 min; HF cache may add time)
# Run after cell 4 if the server is already up. To run standalone (no cell 4), use --launch-server instead.
%run training/smoke_colab.py --base-url http://127.0.0.1:7860

In [ ]:
# 6) Full training (mirrors training/train.py — requires GPU runtime)
%run training/train.py --base-url http://127.0.0.1:7860

In [ ]:
# 7) Plot reward curve inline (after training)
from pathlib import Path
from IPython.display import Image, display

p = Path("assets/reward_curve.png")
if p.is_file():
    display(Image(filename=str(p)))
else:
    print("No assets/reward_curve.png yet — run training first.")

In [ ]:
# 8) Before/after — short transcripts (needs Unsloth + GPU for 7B demo) + plot if present
from pathlib import Path

from IPython.display import Image, display
from client.client import BurenClient
from environment.curriculum import CurriculumManager
from training.prompt_utils import chat_prompt_token_ids, parse_response
import torch

client = BurenClient("http://127.0.0.1:7860")
cur = CurriculumManager()

try:
    from unsloth import FastLanguageModel
except Exception:
    print("[Colab] Unsloth not usable — skip 7B demo. Training uses HF+PEFT fallback.")
else:
    model, tokenizer = FastLanguageModel.from_pretrained(
        "unsloth/Qwen2.5-7B-Instruct", max_seq_length=2048, load_in_4bit=True
    )
    FastLanguageModel.for_inference(model)
    device = next(model.parameters()).device

    def transcript(ep_seed, label):
        obs = client.reset(seed=ep_seed, starting_age=cur.get_starting_age())
        lines = [f"=== {label} ep={ep_seed} ==="]
        steps = 0
        while not obs.done and steps < 3:
            lines.append("--- scenario ---")
            lines.append(obs.scenario_text[:800])
            messages = [{"role": "user", "content": obs.prompt}]
            pids = chat_prompt_token_ids(tokenizer, messages)
            input_ids = torch.tensor([pids], device=device)
            with torch.no_grad():
                out = model.generate(
                    input_ids=input_ids,
                    max_new_tokens=400,
                    temperature=0.8,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
            text = tokenizer.decode(out[0].tolist()[len(pids) :], skip_special_tokens=True)
            lines.append("--- model ---")
            lines.append(text[:1200])
            act = parse_response(text)
            obs, r, d = client.step(act)
            lines.append(f"reward={r:.3f} done={d}")
            steps += 1
        print("\n".join(lines))

    transcript(1, "Demo A")
    transcript(2, "Demo B")

p = Path("assets/before_after.png")
if p.is_file():
    display(Image(filename=str(p)))
else:
    print("No assets/before_after.png yet — run training first.")